# Why #1 Is #1 — ARC-AGI-3 Gap Analysis

*By cosmorin. One of a six-part series: for each competition we enter, we compare the top of the leaderboard with the public frontier, separate what is **documented fact** from what is **inference**, and finish with concrete moves a reader can use to close the gap.*


## The gap in one paragraph

ARC-AGI-3 asks an agent to walk into an unknown game, discover its rules by playing, and clear it efficiently (the score, RHAE, rewards clearing levels with fewer actions than a human baseline). First place is **71.48**, second **68.91**, the gold line **38.22** — and our own best is **23.39** (~1,793rd). Unusually for Kaggle, the winning *ideas* here are largely public: the Milestone-1 and Milestone-2 winning harnesses are written up by the organisers. So why is the gap still ~48 points? Our answer, backed by an embarrassing amount of first-hand measurement: **in this competition the harness is the score** — time allocation, memory, and serving geometry move tens of points while the model stays identical. We proved this the hard way, by breaking a 45-point harness down to 10 points with a single scheduling formula.

✅ = documented fact (leaderboard, organiser write-ups, repositories, our measured runs). 🔍 = inference.


## Leaderboard snapshot

✅ *Downloaded via the Kaggle CLI on 2026-10-25, ~13:59 UTC.*

| Rank | Team | Score (RHAE) | Submissions |
|---|---|---|---|
| 1 | Yi-Chia Chen @ Fields Model | **71.48** | 41 |
| 2 | Tufa Labs | 68.91 | 171 |
| 3 | Team Turducken | 56.02 | 209 |
| 8 | NVARC3 | 45.90 | 46 |
| 16 | Daniel Franzen | 38.85 | 106 |
| 19 (gold line) | Advaith Vecham | 38.22 | 55 |
| ~1,793 | **cosmorin (us)** | **23.39** | — |

Context: at the March 2026 launch, frontier API models scored 0.25–0.37% on this benchmark. Everything above ~30 on this board is scaffolding, not a smarter base model.


## What the public record shows

✅ **Milestone 1 was won by a harness, and the harness is open.** Tufa Labs' "The Duck" ([organiser write-up](https://arcprize.org/blog/arc-prize-2026-milestone-1); code: [duck-harness](https://github.com/Tufalabs/duck-harness)) lets a small local LLM (Qwen 3.6 27B FP8) play by *writing and running Python in a live REPL*, perceiving the board three ways (rendered image, raw ASCII grid, a segmentation tool for zooming), and evicting the oldest context so it can play indefinitely. The team reports that hand-crafted tools *hurt*; the gains came from multimodality and better base models. The other M1 winners used plain JSON actions plus memory tricks — 2nd place Reki: a **reflection memory** refreshed every ~10 steps and a **dead-signature** list (object types that don't respond to clicks are never clicked again in that level). Notably, the 3rd-place team's best run had all its extra mechanisms switched **off**.

✅ **Milestone 2 was won by iterating on exactly that.** Daniel Franzen took the Duck approach and won with faster inference, better compute scheduling *across games*, and tools for the agent to track its own actions (organiser announcement, summarised in [TechTimes' coverage](https://www.techtimes.com/articles/328542/20261006/arc-agi-3-scaffolding-beats-model-upgrades-same-ai-two-settings-49x-score-gain.htm)). His 38.85 on today's board is the visible descendant of that work, and the public Franzen lineage clusters at LB ≈ 39–50.

✅ **Nobody at the top is updating weights inside the run.** All Milestone-1 winners froze their models; the "learning" lives in memory layers. We audited Tufa's pinned research repos first-hand: [ladder](https://github.com/Tufalabs/ladder) is a *mathematics* variant generator with no RL loop, and [opsd-predictive-law](https://github.com/Tufalabs/opsd-predictive-law) is batch post-training code for math/code benchmarks (Apache-2.0) — neither is an in-run ARC learner. The 68.91 Duck's method remains unpublished.

✅ **The strongest public evidence for "what else works" is a memory layer, not training.** Prime Intellect's Continual Harness / Prime Agent ([github.com/PrimeIntellect-ai/prime-agent](https://github.com/PrimeIntellect-ai/prime-agent)) keeps weights frozen while the agent rewrites its own prompts, memory and importable Python *skills* across trajectories — reported 30% → 95.5% on the public ARC-AGI-3 set with a frontier API model. Bento's internal study measured a memory layer alone multiplying a fixed harness's score by ~2.6× (low absolute base).


## What our own runs measured (the expensive part)

We run a Duck-lineage harness (Qwen3.8-Flash-Next, W4A16, served by SGLang on one RTX PRO 6000). Our reproduction of the public baseline averages **45.32** across the 25 public games × 4 passes. Then:

✅ **We broke it to 10.24 without touching the model.** Our Phase-2 v4 added a dynamic time allocator. Root-cause analysis (code + the allocator's own 944-tick log, exact match) showed its formula divided the *total remaining budget* by the number of live sessions, collapsing each session's effective deadline from a flat ~29,500 s to **900 s**. 96 of 100 runs died ~100 minutes in; generation fell from 17.10M to 5.12M tokens. Same weights, same server, same games: **−35 points from one scheduling equation.** A fixed-configuration validation run (allocator off, baseline geometry restored) survived every run to its full cap and the early-death signature disappeared.

✅ **The loss autopsy on 100 baseline runs (45.32 avg, 32 full clears, 401 levels):** the dominant loss is *depth truncation* — in 100/100 runs, zero actions were spent beyond the first un-cleared level. Losing runs burn ~2× the tokens/actions of winning runs (1,584 vs 760 tokens/action). Agents typically give up at ~0.67× the human action budget (median), and wall-clearing breakthroughs happen at up to 1.14× — quitting "on time" is systematically too early.

✅ **Skill extraction from 100 trajectories** produced a sharp separator: median tokens/action is 689 on clears vs 1,421 on stalls (2.06×). "Board changed" is not progress — during stalls the unique-board rate stays 0.96 while nothing is being learned. The highest-value generic skill we distilled: *when stalled, switch to cheap probe actions instead of generating longer deliberations.*

✅ **In-run weight updates (TTT) are a trap in this envelope, and we can price it.** Restarting the inference server costs ~17 minutes and wipes all KV prefixes; inference alone already saturates the ~9 h cap (~19.4M tokens needed vs the cap); and success-imitation training cannot teach the levels you are *failing* — which is where the loss is. The published winners agree by omission: none of them do it.


## Why the gap exists (inference)

🔍 **#1 (71.48) most likely stacks three public ideas better than anyone else, on a stronger serving setup**: a code-writing agent (Duck pattern), a cross-level/cross-game memory-and-skill layer (Continual-Harness pattern), and ruthless compute scheduling (Franzen pattern). None of these require secret science — the public artefacts describe each piece. The gap is *integration quality and iteration count* (note the submission counts: 171 for Tufa, 209 for Turducken).

🔍 **The steep drop from 71 → 38 (gold) → 23 (us)** suggests most of the field has exactly one of the three layers. A JSON-action agent with no persistent memory plateaus early; a REPL agent with broken time allocation (we measured this!) scores like a much dumber model.

## How to close the gap (a reader's checklist)

1. **Fix time allocation before anything else.** Flat, generous per-session caps beat clever dynamic shares. Our A/B is brutal: identical stack, 10.24 vs 45.32. Log every session's wall-clock and kill any scheduler whose deadlines shrink as more sessions register.
2. **Adopt the code-writing REPL pattern** with multi-channel perception (image + ASCII + segmentation zoom), and let the model improvise tools rather than hand-crafting them (Tufa's reported finding).
3. **Add a memory layer with teeth**: pin verified rules outside the evictable history, keep an exclusion list (dead-signatures), and distil cleared levels into short `WHEN…DO…→…` skills carried into later levels.
4. **Instrument token economics.** Track tokens/action on clears vs stalls; when the ratio degrades, force cheap probes. Your scarcest resource is not actions — it is *deliberation spent on a wall*.
5. **Don't do in-run training.** Spend the same effort on offline screening: measure the student/teacher gap on your own transcripts before dreaming of self-distillation (the one freely reusable idea in Tufa's OPSD paper).

## The honest bottom line

✅ ARC-AGI-3 is the rare leaderboard where the winners published the recipe and the gap *persisted* — because the recipe is an engineering discipline (scheduling, memory, serving), not a prompt. 🔍 The teams at 68–71 are not using different science; they are using the same science with fewer self-inflicted wounds.

---

**Credits & sources.** Leaderboard: Kaggle CLI snapshot, 2026-10-25. ARC Prize Milestone 1 write-up: [arcprize.org/blog/arc-prize-2026-milestone-1](https://arcprize.org/blog/arc-prize-2026-milestone-1). Tufa Labs: [duck-harness](https://github.com/Tufalabs/duck-harness), [ladder](https://github.com/Tufalabs/ladder), [opsd-predictive-law](https://github.com/Tufalabs/opsd-predictive-law) (Apache-2.0). Prime Agent: [github.com/PrimeIntellect-ai/prime-agent](https://github.com/PrimeIntellect-ai/prime-agent). Milestone 2 (Franzen) summary: [TechTimes](https://www.techtimes.com/articles/328542/20261006/arc-agi-3-scaffolding-beats-model-upgrades-same-ai-two-settings-49x-score-gain.htm). Unpublished measurements (v4 root cause, loss autopsy, skill extraction, TTT costing) are our own runs, 2026-10-24/25.
